In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/search_interactions_features.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 50000
Columns: 19


,event_id,user_id,session_id,query_text,timestamp,result_position,clicked,dwell_time_seconds,num_results_viewed,query_reformulated,session_duration,country,device_type,is_suspicious,short_dwell_flag,high_reformulation_flag,low_results_viewed_flag,rapid_interaction_flag,behavior_risk_score
0,E000001,U00861,S00838,mathematics,2026-01-01 00:00:00,4,0,9.80,2,0,349,india,india,0,1,0,1,0,2
1,E000002,U03773,S01893,statistics tutorial,2026-01-01 00:10:00,2,0,165.02,1,0,418,india,india,0,0,0,1,0,1
2,E000003,U03093,S01389,data analytics,2026-01-01 00:20:00,9,1,39.43,8,0,599,UK,UK,0,0,0,0,0,0
3,E000004,U00467,S07166,python tutorial,2026-01-01 00:30:00,9,0,59.33,10,0,371,india,india,0,0,0,0,0,0
4,E000005,U04427,S10134,machine learning,2026-01-01 00:40:00,2,1,75.18,2,1,887,india,india,0,0,1,1,0,2


In [2]:
# Target variable
y = df["is_suspicious"]

# Features for the ML model
feature_columns = [
    "dwell_time_seconds",
    "num_results_viewed",
    "query_reformulated",
    "session_duration",
    "short_dwell_flag",
    "high_reformulation_flag",
    "low_results_viewed_flag",
    "rapid_interaction_flag"
]

X = df[feature_columns]

print("Features:", X.shape)
print("Target:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features: (50000, 8)
Target: (50000,)

Target distribution:
is_suspicious
0    46078
1     3922
Name: count, dtype: int64


In [6]:
X_np = X.to_numpy(dtype=float)
y_np = y.to_numpy(dtype=float)

# Standardize features
mean = X_np.mean(axis=0)
std = X_np.std(axis=0)

std[std == 0] = 1

X_scaled = (X_np - mean) / std

# Add intercept
X_scaled = np.column_stack([
    np.ones(len(X_scaled)),
    X_scaled
])

# Train/test split manually
np.random.seed(42)

indices = np.random.permutation(len(X_scaled))

split = int(0.8 * len(indices))

train_idx = indices[:split]
test_idx = indices[split:]

X_train = X_scaled[train_idx]
X_test = X_scaled[test_idx]

y_train = y_np[train_idx]
y_test = y_np[test_idx]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (40000, 9)
Testing data: (10000, 9)


In [7]:
# Logistic Regression using NumPy

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

# Initialize weights
weights = np.zeros(X_train.shape[1])

learning_rate = 0.05
epochs = 1000

for epoch in range(epochs):
    # Predictions
    probabilities = sigmoid(X_train @ weights)

    # Gradient
    gradient = (X_train.T @ (probabilities - y_train)) / len(y_train)

    # Update weights
    weights -= learning_rate * gradient

    # Print progress
    if (epoch + 1) % 200 == 0:
        loss = -np.mean(
            y_train * np.log(probabilities + 1e-9)
            + (1 - y_train) * np.log(1 - probabilities + 1e-9)
        )
        print(f"Epoch {epoch + 1}: Loss = {loss:.4f}")

print("\nModel training completed!")

Epoch 200: Loss = 0.1579
Epoch 400: Loss = 0.1041
Epoch 600: Loss = 0.0831
Epoch 800: Loss = 0.0715
Epoch 1000: Loss = 0.0640

Model training completed!


In [8]:
# Evaluate the model

test_probabilities = sigmoid(X_test @ weights)

# Convert probabilities to 0/1 predictions
y_pred = (test_probabilities >= 0.5).astype(int)

# Confusion matrix values
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

# Metrics
accuracy = (TP + TN) / len(y_test)

precision = TP / (TP + FP) if (TP + FP) > 0 else 0

recall = TP / (TP + FN) if (TP + FN) > 0 else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("Confusion Matrix:")
print("TN:", TN, "FP:", FP)
print("FN:", FN, "TP:", TP)

print("\nModel Performance:")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

Confusion Matrix:
TN: 9156 FP: 78
FN: 1 TP: 765

Model Performance:
Accuracy : 0.9921
Precision: 0.9075
Recall   : 0.9987
F1-score : 0.9509


In [9]:
# Calculate ROC-AUC manually

scores = test_probabilities

# Sort scores from highest to lowest
order = np.argsort(-scores)

sorted_y = y_test[order]

# Number of positive and negative samples
n_positive = np.sum(sorted_y == 1)
n_negative = np.sum(sorted_y == 0)

# Rank-based AUC calculation
positive_ranks = np.where(sorted_y == 1)[0] + 1

auc = (
    positive_ranks.sum()
    - n_positive * (n_positive + 1) / 2
) / (n_positive * n_negative)

print("ROC-AUC:", round(auc, 4))

ROC-AUC: 0.001


In [10]:
# Correct ROC-AUC calculation

scores = test_probabilities
labels = y_test.astype(int)

# Sort scores from LOW to HIGH
order = np.argsort(scores)
sorted_scores = scores[order]
sorted_labels = labels[order]

# Assign ranks, handling ties
ranks = np.empty(len(sorted_scores), dtype=float)

i = 0
while i < len(sorted_scores):
    j = i + 1

    while j < len(sorted_scores) and sorted_scores[j] == sorted_scores[i]:
        j += 1

    average_rank = (i + 1 + j) / 2
    ranks[i:j] = average_rank

    i = j

# Sum ranks for positive class
positive_ranks = ranks[sorted_labels == 1]

n_positive = np.sum(labels == 1)
n_negative = np.sum(labels == 0)

# Mann-Whitney based AUC
auc = (
    positive_ranks.sum()
    - n_positive * (n_positive + 1) / 2
) / (n_positive * n_negative)

print("ROC-AUC:", round(auc, 4))

ROC-AUC: 0.999


In [15]:
# Automated Risk Monitoring

monitoring_df = pd.DataFrame(X_test)

print("Monitoring records:", len(monitoring_df))
print("Test features:", monitoring_df.shape)

# Model risk probability
monitoring_df["ml_risk_probability"] = test_probabilities

print("Monitoring setup successful!")

Monitoring records: 10000
Test features: (10000, 9)
Monitoring setup successful!


In [18]:
# Automated Risk Alert
monitoring_df["automated_alert"] = (
    monitoring_df["ml_risk_probability"] >= 0.80
).astype(int)

print("Automated alerts:", monitoring_df["automated_alert"].sum())

print(
    "Alert rate:",
    round(monitoring_df["automated_alert"].mean() * 100, 2),
    "%"
)

Automated alerts: 534
Alert rate: 5.34 %


In [24]:
# Clean automated monitoring

monitoring_df["automated_alert"] = (
    monitoring_df["ml_risk_probability"] >= 0.80
).astype(int)

print("Monitoring records:", len(monitoring_df))
print("Automated alerts:", monitoring_df["automated_alert"].sum())

print(
    "Alert rate:",
    round(monitoring_df["automated_alert"].mean() * 100, 2),
    "%"
)

Monitoring records: 10000
Automated alerts: 534
Alert rate: 5.34 %


In [25]:
# Save automated monitoring results

output_path = "../data/processed/automated_monitoring_results.csv"

monitoring_df.to_csv(output_path, index=False)

print("Monitoring results saved successfully!")
print(output_path)

Monitoring results saved successfully!
../data/processed/automated_monitoring_results.csv


In [26]:
# Create clean dashboard dataset

dashboard_df = df.iloc[test_idx].copy()

dashboard_df["ml_risk_probability"] = test_probabilities
dashboard_df["actual_label"] = y_test.astype(int)
dashboard_df["predicted_label"] = y_pred.astype(int)

# Automated alert at 0.80 threshold
dashboard_df["automated_alert"] = (
    dashboard_df["ml_risk_probability"] >= 0.80
).astype(int)

# Save dashboard-ready dataset
output_path = "../data/processed/search_quality_dashboard.csv"
dashboard_df.to_csv(output_path, index=False)

print("Dashboard dataset created successfully!")
print("Rows:", len(dashboard_df))
print("Columns:", len(dashboard_df.columns))
print(output_path)

Dashboard dataset created successfully!
Rows: 10000
Columns: 23
../data/processed/search_quality_dashboard.csv
